<a href="https://colab.research.google.com/github/viettungvuong/Distillation-Playground/blob/main/Distillation_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!pip install -q peft bitsandbytes accelerate

**Huggingface login**

In [19]:
from huggingface_hub import login

login()

**Downloading dataset**

In [22]:
from datasets import load_dataset

dataset = load_dataset("OpenHust/vietnamese-summarization")

In [23]:
from transformers import AutoProcessor, AutoModelForCausalLM

model_name = "Qwen/Qwen3.6-27B"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig
import torch

# 1. Setup 4-bit quantization to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Re-load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 3. Prepare model for kbit training (gradient checkpointing, etc.)
model = prepare_model_for_kbit_training(model)

# 4. Define LoRA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Target linear layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 5. Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


**Data processing**

In [24]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
        num_rows: 74564
    })
})

In [25]:
split_dataset = dataset["train"].train_test_split(test_size=0.2)

train_set = split_dataset['train']
test_set = split_dataset['test']

In [26]:
train_set[:5]

{'Unnamed: 0': [10616, 8486, 508, None, 5366],
 'Document': ['Tính cách\nCho Spaniel Pháp có một tính cách thân thiện và cởi mở có tính cân bằng và kiên nhẫn. Nó không phải là một con chó có bản tính hung dữ, háo hức muốn làm hài lòng chủ nhân và do đó có thể được huấn luyện cách dễ dàng. Chó thuộc giống chó này sẽ tạo thành một liên kết mạnh mẽ với chủ nhân của nó, thường là một con chó lao động. Nó có sức chịu đựng cao và đòi hỏi phải được rèn luyện cách cứng rắn.\n\n\n== Tham khảo ==',
  'Không thay áo trước mặt em bé. Tránh tắm chung với em bé. Nếu nhìn thấy vú mẹ, trẻ sẽ nhớ lại thói quen cũ và đòi bú lại. Cố gắng không bế trẻ ở tư thế giống như trước đây thường cho trẻ bú. Bế trẻ ở tư thế khác để đánh lạc hướng cơn thèm của trẻ. Tránh ngồi chiếc ghế trước đây bạn vẫn thường hay ngồi cho con bú. Tránh cho con vào căn phòng trước kia bạn vẫn hay vào cho con bú mỗi ngày. Cố gắng thay đổi mọi thói quen có thể “gợi thèm” trẻ.',
  'Bác sĩ N Hà Đức Thiện - khoa N gây mê N hồi sức V - ch

In [34]:
def normalize_text(row):
    # Only handle lowercase normalization
    row['Document'] = str(row['Document']).lower()
    row['Summary'] = str(row['Summary']).lower()
    return row

def add_special_tokens(row):
    # Qwen follows the ChatML format:
    # <|im_start|>user
    # {Document}<|im_end|>
    # <|im_start|>assistant
    # {Summary}<|im_end|>

    formatted_input = (
        f"<|im_start|>\n"
        f"{row['Document']}<|im_end|>\n"
        f"<|im_start|>\n"
        f"{row['Summary']}<|im_end|>"
    )

    row['Processed_Input'] = formatted_input
    return row

# Apply normalization
train_set = train_set.map(normalize_text)
test_set = test_set.map(normalize_text)

# Apply special tokens and create new column
train_set = train_set.map(add_special_tokens)
test_set = test_set.map(add_special_tokens)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [36]:
print("Special tokens: ", tokenizer.special_tokens_map)
sep_token_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
print(f"Sep token is {sep_token_id}")

Special tokens:  {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}
Sep token is 248045


In [37]:
def tokenize_function(batch):
    # Tokenize the processed input
    model_inputs = tokenizer(
        batch["Processed_Input"],
        truncation=True,
        max_length=1542,
        padding='max_length'
    )

    labels = []

    for input_ids in model_inputs["input_ids"]:
        label = list(input_ids)
        try:
            # Find the index of the first separation token
            indices = [i for i, val in enumerate(label) if val == sep_token_id]
            turn_model_idx = indices[-1]
            # Set all tokens up to the first separation to -100 (loss function ignore)
            for i in range(turn_model_idx + 1):
                label[i] = -100
        except ValueError as e:
            print(e)

        labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs


In [38]:
tokenized_train = train_set.map(tokenize_function, batched=True, remove_columns=train_set.column_names)
tokenized_test = test_set.map(tokenize_function, batched=True, remove_columns=test_set.column_names)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

**Fine-tuning**

In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="summarization",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [40]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

[transformers] The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
# Re-initialize trainer to pick up the LoRA wrapped model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [41]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None, 'pad_token_id': 248044}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 76.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 10.81 MiB is free. Including non-PyTorch memory, this process has 79.23 GiB memory in use. Of the allocated memory 73.69 GiB is allocated by PyTorch, and 5.03 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)